In [ ]:
!pip install openai

In [6]:
import openai

openai.__version__

'2.29.0'

In [ ]:
!pip install dotenv

In [3]:
from dotenv import load_dotenv
from openai import OpenAI


load_dotenv(override=True)  # .env 파일 로드
client = OpenAI()
client

print(client)

여기 수정된 버전입니다.

---

**messages**
AI와 나눈 대화 내역입니다. 사용자와 AI의 메시지를 순서대로 담아 전달합니다.

**max_tokens**
AI가 한 번에 생성할 수 있는 최대 글자량(토큰)입니다. 너무 작으면 답변이 중간에 잘립니다.

**n**
같은 질문에 대해 몇 가지 답변을 생성할지 설정합니다. 기본값 1을 권장합니다. 숫자가 클수록 비용도 그만큼 늘어납니다.

**response_format**
응답 형식을 지정합니다. `{ "type": "json_object" }` 로 설정하면 JSON 형태로만 답변을 받을 수 있습니다.

**temperature**
답변의 창의성(무작위성)을 조절합니다. 0에 가까울수록 일관되고 정확한 답변, 2에 가까울수록 다양하고 창의적인 답변이 나옵니다. `top_p`와 동시에 조절하지 않는 것을 권장합니다.

**top_p**
temperature와 비슷하게 답변의 다양성을 조절하는 또 다른 방법입니다. 값이 낮을수록 확률이 높은 단어만 선택합니다. temperature와 둘 중 하나만 조절하는 것을 권장합니다.

In [9]:
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": "당신은 파이썬 프로그래머입니다.",
        },
        {
            "role": "user",
            "content": "피보나치 수열을 생성하는 파이썬 프로그램을 작성해주세요.",
        },
    ],
    max_tokens=1024,        # 코드 생성이므로 충분한 길이 확보
    n=1,                    # 답변 1개만 생성 (비용 절감)
    temperature=0.2,        # 낮게 설정 → 정확하고 일관된 코드 생성
    top_p=0.9,              # temperature 보조, 너무 편향되지 않게
    response_format={"type": "text"},  # 코드+설명 혼합이므로 text 형식
)

print(completion.choices[0].message.content)

물론입니다! 아래는 피보나치 수열을 생성하는 파이썬 프로그램입니다.

```python
def fibonacci(n):
    fib_sequence = [0, 1]
    for i in range(2, n):
        next_num = fib_sequence[i-1] + fib_sequence[i-2]
        fib_sequence.append(next_num)
    return fib_sequence

n = int(input("피보나치 수열의 길이를 입력하세요: "))
fibonacci_sequence = fibonacci(n)
print(fibonacci_sequence)
```

이 프로그램은 사용자로부터 피보나치 수열의 길이를 입력받아 해당 길이만큼의 피보나치 수열을 생성하고 출력합니다.


In [22]:
print(completion.choices[0].message.content)

물론이죠! 아래는 재귀 함수를 사용하여 피보나치 수열을 생성하는 파이썬 프로그램입니다.

```python
def fibonacci(n):
    if n <= 1:
        return n
    else:
        return fibonacci(n-1) + fibonacci(n-2)

n = int(input("몇 개의 피보나치 수를 생성하시겠습니까? "))

if n <= 0:
    print("유효하지 않은 입력 값입니다. 0보다 큰 정수를 입력해주세요.")
else:
    print("피보나치 수열:")
    for i in range(n):
        print(fibonacci(i))
```

이 프로그램은 사용자로부터 몇 개의 피보나치 수를 생성할 지 입력받은 뒤, 해당 개수만큼의 피보나치 수열을 출력합니다.


2. 스트리밍 

In [ ]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4-turbo-preview",
    messages=[
        {
            "role": "system",
            "content": "당신은 파이썬 프로그래머입니다.",
        },
        {
            "role": "user",
            "content": "피보나치 수열을 생성하는 파이썬 프로그램을 작성해주세요.",
        },
    ],
    stream=True,  # 스트림 모드 활성화
)

final_answer = []

# 스트림 모드에서는 completion.choices 를 반복문으로 순회
for chunk in completion:
    # chunk 를 저장
    chunk_content = chunk.choices[0].delta.content
    # chunk 가 문자열이면 final_answer 에 추가
    if isinstance(chunk_content, str):
        # final_answer.append(chunk_content)
        # # 토큰 단위로 실시간 답변 출력
        # print(chunk_content, end="")
        # 전체 답변인 final_answer 를 문자열로 변환하여 출력
        final_answer = "".join(final_answer)
        print(final_answer)

요즘은 아래 처럼 하기도한다

In [2]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4-turbo-preview",
    input="Write a one-sentence bedtime story about a unicorn."
)

print(response.output_text)

In the heart of a magical forest bathed in moonlight, a young unicorn with a shimmering, iridescent mane embarked on a nocturnal adventure, discovering a hidden glen where dreams take flight, forever changing the night into a realm of endless possibilities.


연속대화

client.chat.completions.create() 함수는 자체적으로 대화를 저장하는 기능이 없습니다.

따라서, 이전 문맥(context)을 기억하면서 대화를 이어나가기 위해서는(일반적인 ChatBot을 생각하시면 됩니다) 다음과 같이 messages 옵션에 메시지를 추가해야 합니다.

messages는 대화의 각 부분을 구성하는 데 사용되며, 각 메시지는 "role"과 "content"라는 두 가지 주요 요소를 포함하는 딕셔너리 형태로 되어 있습니다. 이 구조를 사용하면 복잡한 대화 흐름을 더 명확하게 관리하고 조작할 수 있습니다.

각 "role"의 의미는 다음과 같습니다:

"system": 시스템 전역 설정에 대한 지시문을 포함합니다. 예를 들어, 모델에 특정한 페르소나(persona)를 부여하거나, 대화의 맥락을 설정하는 데 사용됩니다.
"user": 사용자의 입력을 나타냅니다. 이는 대화에서 사용자가 질문하거나 요청한 내용을 담고 있습니다.
"assistant": AI 모델(예: ChatGPT)의 응답을 나타냅니다. 이는 모델이 생성한 답변이나 정보를 포함합니다.

In [4]:
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. You must answer in Korean.",
        },
        {
            "role": "user",
            "content": "대한민국의 수도는 어디인가요?",
        },
    ],
)

print(completion.choices[0].message.content)

대한민국의 수도는 서울입니다.


In [5]:
def ask(question):
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. You must answer in Korean.",
            },
            {
                "role": "user",
                "content": question,  # 사용자의 질문을 입력
            },
        ],
    )
    # 답변을 반환
    return completion.choices[0].message.content

In [6]:
# 첫 번째 질문
ask("대한민국의 수도는 어디인가요?")

'대한민국의 수도는 서울입니다.'

In [7]:
# 두 번째 질문
ask("영어로 답변해 주세요")

'저는 한국어로만 응답할 수 있습니다. 궁금한 내용이 있으면 언제든지 물어보세요!'

위의 답변에서 보듯이, GPT가 다음 질문에 대한 답변을 잘못했습니다. 이는 이전 대화내용에 대한 저장을 하지 않았기 때문 입니다.

이를 해결하기 위해, 대화의 연속성을 유지하는 로직을 추가해야 합니다. 대화의 각 부분(사용자의 질문과 AI의 응답)을 messages 리스트에 순차적으로 추가함으로써, 챗봇은 이전의 대화 내용을 참조하여 적절한 답변을 생성할 수 있습니다.

In [8]:
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. You must answer in Korean.",
        },
        {
            "role": "user",
            "content": "대한민국의 수도는 어디인가요?",  # 첫 번째 질문
        },
        {
            "role": "assistant",
            "content": "대한민국의 수도는 서울입니다.",  # 첫 번째 답변
        },
        {
            "role": "user",
            "content": "이전의 답변을 영어로 번역해 주세요.",  # 두 번째 질문
        },
    ],
)

# 두 번째 답변을 출력
print(completion.choices[0].message.content)

The capital of South Korea is Seoul.


좀 더 깔끔하게 함수화하여 구현해 보겠습니다.

In [9]:
def ask(question, message_history=[], model="gpt-3.5-turbo"):
    if len(message_history) == 0:
        # 최초 질문
        message_history.append(
            {
                "role": "system",
                "content": "You are a helpful assistant. You must answer in Korean.",
            }
        )

    # 사용자 질문 추가
    message_history.append(
        {
            "role": "user",
            "content": question,
        },
    )

    # GPT에 질문을 전달하여 답변을 생성
    completion = client.chat.completions.create(
        model=model,
        messages=message_history,
    )

    # 사용자 질문에 대한 답변을 추가
    message_history.append(
        {"role": "assistant", "content": completion.choices[0].message.content}
    )

    return message_history

In [10]:
# 최초 질문
message_history = ask("양자역학에 대해서 쉽게 설명해 주세요", message_history=[])
# 최초 답변
print(message_history[-1])

{'role': 'assistant', 'content': '양자역학은 아주 작은 입자들에 대한 묘사를 다루는 물리 이론입니다. 이 이론에 따르면 입자들은 확률적으로 움직이며, 우리가 알 수 있는 것보다도 놀라운 특성을 가지고 있습니다. 예를 들어 양자역학에 의하면 입자들은 동시에 파동과 입자의 성질을 모두 가지고 있을 수 있습니다. 또한 양자역학은 매우 정확한 실험 결과를 예측하며, 많은 현대 기술과 기술의 발전에 큰 역할을 합니다.'}


In [11]:
message_history

[{'role': 'system',
  'content': 'You are a helpful assistant. You must answer in Korean.'},
 {'role': 'user', 'content': '양자역학에 대해서 쉽게 설명해 주세요'},
 {'role': 'assistant',
  'content': '양자역학은 아주 작은 입자들에 대한 묘사를 다루는 물리 이론입니다. 이 이론에 따르면 입자들은 확률적으로 움직이며, 우리가 알 수 있는 것보다도 놀라운 특성을 가지고 있습니다. 예를 들어 양자역학에 의하면 입자들은 동시에 파동과 입자의 성질을 모두 가지고 있을 수 있습니다. 또한 양자역학은 매우 정확한 실험 결과를 예측하며, 많은 현대 기술과 기술의 발전에 큰 역할을 합니다.'}]

In [12]:
# 두 번째 질문
message_history = ask(
    "이전의 내용을 영어로 답변해 주세요", message_history=message_history
)
# 두 번째 답변
print(message_history[-1])

{'role': 'assistant', 'content': 'Quantum mechanics is a physics theory that describes the behavior of very small particles. According to this theory, particles move probabilistically and have surprising properties beyond what we can observe. For example, according to quantum mechanics, particles can simultaneously possess both wave and particle properties. Furthermore, quantum mechanics can predict very accurate experimental results and has played a significant role in the advancement of many modern technologies and sciences.'}


In [13]:
message_history

[{'role': 'system',
  'content': 'You are a helpful assistant. You must answer in Korean.'},
 {'role': 'user', 'content': '양자역학에 대해서 쉽게 설명해 주세요'},
 {'role': 'assistant',
  'content': '양자역학은 아주 작은 입자들에 대한 묘사를 다루는 물리 이론입니다. 이 이론에 따르면 입자들은 확률적으로 움직이며, 우리가 알 수 있는 것보다도 놀라운 특성을 가지고 있습니다. 예를 들어 양자역학에 의하면 입자들은 동시에 파동과 입자의 성질을 모두 가지고 있을 수 있습니다. 또한 양자역학은 매우 정확한 실험 결과를 예측하며, 많은 현대 기술과 기술의 발전에 큰 역할을 합니다.'},
 {'role': 'user', 'content': '이전의 내용을 영어로 답변해 주세요'},
 {'role': 'assistant',
  'content': 'Quantum mechanics is a physics theory that describes the behavior of very small particles. According to this theory, particles move probabilistically and have surprising properties beyond what we can observe. For example, according to quantum mechanics, particles can simultaneously possess both wave and particle properties. Furthermore, quantum mechanics can predict very accurate experimental results and has played a significant role in the advancement of many modern technologies and sciences.'}]

이번에는 이전 대화내용을 message_history 에 저장하여 전달하였습니다. 이전 대화내용을 저장하면서 message_history 를 통해 대화를 이어나갈 수 있습니다.

## json_object 답변형식

이번에는 GPT 의 출력 형식을 지정하는 방법을 살펴보겠습니다. 다음의 예시는 프롬프트를 활용하여 답변을 JSON 형식으로 받는 예제 입니다.

In [15]:
completion = client.chat.completions.create(
    model="gpt-3.5-turbo-1106",
    messages=[
        {
            "role": "system",
            # 답변 형식을 JSON 으로 받기 위해 프롬프트에 JSON 형식을 지정
            "content": "You are a helpful assistant designed to output JSON. You must answer in Korean.",
        },
        {
            "role": "user",
            "content": "대한민국의 수도는 어디인가요?",
        },
    ],
    response_format={"type": "json_object"},  # 답변 형식을 JSON 으로 지정
)

print(completion.choices[0].message.content)

{"답변": "서울"}


In [16]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo-1106",
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant designed to output JSON.",
        },
        {
            "role": "user",
            "content": "통계를 주제로 4지선다형 객관식 문제를 만들어주세요.",
        },
    ],
    temperature=0.5,
    max_tokens=300,
)
print(response.choices[0].message.content)

{
  "question": "다음 중 통계학의 기초 개념이 아닌 것은 무엇인가요?",
  "options": [
    "평균",
    "중앙값",
    "표준편차",
    "지수함수"
  ]
}


response_format={"type": "json_object"} 으로 출력시 json 형태로 출력되는 것을 확인할 수 있습니다.

json 형식으로 출력 값을 받으면 데이터베이스에 저장하거나 파일형태로 저장하는데 용이합니다.

In [17]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo-1106",
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant designed to output JSON.",
        },
        {
            "role": "user",
            "content": "통계를 주제로 4지선다형 객관식 문제를 만들어주세요. 정답은 index 번호로 알려주세요. "
            "난이도는 [상, 중, 하] 중 하나로 표기해 주세요.",
        },
    ],
    temperature=0.5,
    max_tokens=300,
    n=5,
)

5개 출력

In [18]:
for res in response.choices:
    print(res.message.content)

{
  "question": "다음 중 통계학에서 사용되는 가설검정 방법으로, 귀무가설이 기각될 확률을 나타내는 것은?",
  "options": ["p-value", "표준편차", "상관계수", "평균"],
  "answer_index": 0,
  "difficulty": "중"
}
{
  "question": "다음 중 통계학의 기초 개념이 아닌 것은 무엇인가요?",
  "options": ["평균", "중앙값", "분산", "면적"],
  "answer": 3,
  "difficulty": "하"
}
{
  "question": "다음 중 통계학에서 사용되는 가설검정의 과정 중 가장 먼저 수행되는 단계는 무엇인가요?",
  "options": ["가설 설정", "표본 추출", "가설 검정", "결론 도출"],
  "answer_index": 0,
  "difficulty": "중"
}
{
  "question": "다음 중 통계 용어가 아닌 것은?",
  "options": ["평균", "최대값", "중앙값", "상관관계"],
  "answer": 1,
  "difficulty": "하"
}
{
  "question": "다음 중 통계학의 기본 개념이 아닌 것은 무엇인가요?",
  "options": ["평균", "중앙값", "분산", "표준편차"],
  "answer_index": 2,
  "difficulty": "하"
}
